In [ ]:
!pip install transformers torch soundfile datasets
!pip install git+https://github.com/huggingface/transformers.git

In [ ]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
import torch
import numpy as np

from datasets import load_dataset
from IPython.display import Audio

# Load models
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

# Load speaker embeddings dataset
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")

# Get user input
def get_custom_input():
    from IPython.display import clear_output
    clear_output()

    # Input text
    text = input("Enter your text (max 500 characters):\n")[:500]  # Truncate to 500 chars

    # Input speaker choice
    speaker_idx = int(input("\nChoose speaker (0-7306, try 7306 for female, 0 for male): ") or "7306")

    return text, speaker_idx

# Main function
def generate_custom_speech():
    # Get user inputs
    text, speaker_idx = get_custom_input()

    # Load selected speaker embeddings
    try:
        speaker_embeddings = torch.tensor(embeddings_dataset[speaker_idx]["xvector"]).unsqueeze(0)
    except:
        print("Invalid speaker index! Using default (7306)")
        speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

    # Split text into chunks (better for long text)
    chunks = [text[i:i+200] for i in range(0, len(text), 200)]

    full_audio = []
    for chunk in chunks:
        inputs = processor(text=chunk, return_tensors="pt")
        speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
        full_audio.append(speech.numpy())

    # Combine audio chunks
    final_audio = np.concatenate(full_audio)

    # Play audio
    display(Audio(final_audio, rate=16000))

# Run the generator
generate_custom_speech()